# Notebook 01: Inspección Inicial
## Proyecto Integrador — Minería de Datos I

**Objetivo:** Presentar la estructura general, dimensiones, tipos de datos, valores faltantes, duplicados y observaciones iniciales del dataset. Reunir evidencia para orientar las etapas posteriores.

**Dataset:** `data/raw/reporte_clinica_original.csv` — Reporte clínico de costos de seguro médico.

In [ ]:
# 1. Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 120)

print('Librerías cargadas correctamente.')

In [ ]:
# 2. Carga del dataset
DATA_PATH = '../data/raw/reporte_clinica_original.csv'
df = pd.read_csv(DATA_PATH)

print(f'Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas')

## 2. Estructura general y dimensiones

In [ ]:
# Dimensiones
print(f'Filas:    {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')
print(f'Celdas totales: {df.size}')
print(f'\nMemoria usada:')
print(df.info(memory_usage='deep'))

## 3. Tipos de datos

In [ ]:
# Tipos de datos por columna
tipos = pd.DataFrame({
    'Columna': df.columns,
    'Tipo': df.dtypes.values,
    'No Nulos': df.notnull().sum().values,
    'Nulos': df.isnull().sum().values,
    '% Nulos': (df.isnull().sum() / len(df) * 100).values
})
tipos['% Nulos'] = tipos['% Nulos'].round(2)
tipos

### Observación sobre tipos de datos:
- Columnas numéricas: `age` (float64), `bmi` (float64), `children` (int64), `charges` (float64)
- Columnas categóricas: `sex` (str), `smoker` (str), `region` (str)
- `Unnamed: 0` parece ser un índice sobrante de una exportación previa

## 4. Primeras y últimas filas

In [ ]:
# Primeras 10 filas
print('=== PRIMERAS 10 FILAS ===')
df.head(10)

In [ ]:
# Últimas 10 filas
print('=== ÚLTIMAS 10 FILAS ===')
df.tail(10)

In [ ]:
# Muestra aleatoria de 5 filas
df.sample(5, random_state=42)

## 5. Valores faltantes (nulos)

In [ ]:
# Nulos por columna
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)

print('Columnas con valores nulos:')
for col, n in nulos.items():
    pct = n / len(df) * 100
    print(f'  {col}: {n} nulos ({pct:.2f}%)')

# Visualización
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Mapa de calor de nulos
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='Reds', ax=ax[0])
ax[0].set_title('Mapa de calor de valores nulos')

# Barras de nulos
nulos_all = df.isnull().sum()
nulos_all[nulos_all > 0].plot(kind='bar', ax=ax[1], color='coral', edgecolor='black')
ax[1].set_title('Cantidad de nulos por columna')
ax[1].set_ylabel('Nulos')
ax[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Observación sobre nulos:
- **age**: 102 nulos (7.48% del total) — se deberá decidir entre imputación o eliminación.
- **bmi**: 50 nulos (3.67% del total) — misma situación.
- Las demás columnas no presentan valores nulos.

## 6. Duplicados

In [ ]:
# Verificar duplicados
n_dup = df.duplicated().sum()
n_dup_sin_idx = df.drop(columns=['Unnamed: 0']).duplicated().sum()

print(f'Filas duplicadas (todas las columnas): {n_dup}')
print(f'Filas duplicadas (sin columna índice): {n_dup_sin_idx}')

**Observación:** No se detectan filas duplicadas en el dataset.

## 7. Valores únicos y posibles inconsistencias

In [ ]:
# Análisis de valores únicos por columna
print('=== VALORES ÚNICOS POR COLUMNA ===\n')
for col in df.columns:
    unicos = df[col].dropna().unique()
    n_unico = len(unicos)
    print(f'{col}: {n_unico} valores únicos')
    if n_unico <= 20:
        valores = sorted(unicos, key=str)
        print(f'  → {valores}')
    print()

### Observación sobre valores únicos:

| Columna | Valores | Problema detectado |
|---|---|---|
| `sex` | female, male, Female, Male | **Inconsistencia** de mayúsculas/minúsculas |
| `smoker` | yes, no, Yes, No | **Inconsistencia** de mayúsculas/minúsculas |
| `region` | southwest, SE, southeast, northwest, NE, northeast, SW, NW | **Inconsistencia**: siglas mezcladas con nombres completos |
| `children` | 0, 1, 2, 3, 4, 5, 15, 99, -1 | **Valores atípicos**: 99, 15, -1 fuera de rango |
| `bmi` | (533 valores) | **Rango sospechoso**: valor -5 y 999 detectados |

## 8. Análisis de variables numéricas

In [ ]:
# Estadísticas descriptivas
print('=== ESTADÍSTICAS DESCRIPTIVAS ===\n')
df.describe()

In [ ]:
# Boxplots para detectar outliers visualmente
cols_numericas = ['age', 'bmi', 'children', 'charges']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(cols_numericas):
    ax = axes[i]
    df[col].dropna().plot(kind='box', ax=ax, vert=True, patch_artist=True,
                          boxprops=dict(facecolor='lightblue', alpha=0.7))
    ax.set_title(f'Boxplot: {col}')
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# Detalle de valores extremos en bmi
print('=== VALORES EXTREMOS EN BMI ===')
bmi_extremos = df[(df['bmi'] <= 0) | (df['bmi'] >= 100)]
print(f'BMI ≤ 0: {(df["bmi"] <= 0).sum()} registros')
print(f'BMI ≥ 100: {(df["bmi"] >= 100).sum()} registros')
if len(bmi_extremos) > 0:
    bmi_extremos[['age', 'sex', 'bmi', 'children', 'charges']]

In [ ]:
# Detalle de valores extremos en children
print('=== VALORES EXTREMOS EN CHILDREN ===')
children_extremos = df[(df['children'] < 0) | (df['children'] > 10)]
print(f'children < 0: {(df["children"] < 0).sum()} registros')
print(f'children > 10: {(df["children"] > 10).sum()} registros')
if len(children_extremos) > 0:
    children_extremos[['age', 'sex', 'bmi', 'children', 'charges']]

### Observación sobre outliers:
- **bmi**: Hay 1 registro con valor **-5** (fisiológicamente imposible) y 1 registro con valor **999** (extremadamente atípico).
- **children**: Hay 3 registros con valor **-1** (imposible), 2 registros con **15** y 1 registro con **99** (claramente erróneos).
- **charges**: La distribución muestra un rango amplio ($1,121 – $63,770) lo cual es esperable dado que los costos dependen de múltiples factores.

## 9. Distribución de variables categóricas

In [ ]:
# Variables categóricas
cols_cat = ['sex', 'smoker', 'region']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, col in enumerate(cols_cat):
    counts = df[col].value_counts()
    counts.plot(kind='bar', ax=axes[i], color='steelblue', edgecolor='black')
    axes[i].set_title(f'Distribución: {col}')
    axes[i].set_ylabel('Frecuencia')
    axes[i].tick_params(axis='x', rotation=45)
    
    # Agregar etiquetas
    for j, v in enumerate(counts):
        axes[i].text(j, v + 5, str(v), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# Tabla de frecuencias
print('\n=== FRECUENCIAS DE VARIABLES CATEGÓRICAS ===')
for col in cols_cat:
    print(f'\n{col}:')
    print(df[col].value_counts().to_string())

### Observación sobre variables categóricas:
- **sex**: 4 valores cuando deberían ser 2 → male/Male y female/Female deben unificarse.
- **smoker**: 4 valores cuando deberían ser 2 → yes/Yes y no/No deben unificarse.
- **region**: 8 valores cuando deberían ser 4 → SE/southeast, NE/northeast, SW/southwest, NW/northwest deben unificarse.

## 10. Formulación de preguntas de análisis

A partir de esta inspección inicial, formulamos las siguientes preguntas que guiarán el análisis exploratorio:

1. **¿Cómo se relaciona la edad con el costo del seguro médico?**
2. **¿Existen diferencias significativas en los costos entre fumadores y no fumadores?**
3. **¿El IMC (bmi) es un factor determinante en el costo del seguro?**
4. **¿Hay diferencias regionales en los costos de seguro?**
5. **¿El número de hijos influye en el costo del seguro?**
6. **¿Qué combinación de factores explica mejor la variabilidad en los costos?**

Cada pregunta será respondida con evidencia obtenida durante el EDA y PCA.

## 11. Resumen de hallazgos de la inspección inicial

### Dataset
- **1,363 registros** y **8 columnas** sobre costos de seguro médico.
- Columnas: índice (`Unnamed: 0`), edad, sexo, IMC, hijos, fumador, región, costo.

### Problemas identificados para la etapa de limpieza

| # | Problema | Columnas afectadas | Cantidad |
|---|---|---|---|
| 1 | Valores nulos | `age`, `bmi` | 152 total |
| 2 | Inconsistencia mayúsculas/minúsculas | `sex`, `smoker` | 4 variantes c/u |
| 3 | Siglas vs nombres completos | `region` | 8 variantes (deberían ser 4) |
| 4 | Valores fuera de rango | `bmi` | 2 (-5 y 999) |
| 5 | Valores fuera de rango | `children` | 6 (-1, 15, 99) |
| 6 | Columna índice redundante | `Unnamed: 0` | Toda la columna |

### Próximos pasos
1. **Notebook 02**: Limpieza y preparación — tratar nulos, estandarizar categóricas, corregir outliers.
2. **Notebook 03**: Análisis exploratorio (EDA) con visualizaciones.
3. **Notebook 04**: Reducción de dimensionalidad (PCA).
4. **Notebook 05**: Conclusiones finales.